# 10. LSTM
**Use case:** Supply Chain — Weekly SKU demand forecasting for a distribution center

A distribution center restocks thousands of SKUs every week and buyers order from forecasts built on spreadsheets and 'same week last year'. Holiday peaks and promotions are badly predicted, so the DC runs out during promotions and carries too much stock afterwards. The company has several years of weekly shipments per SKU, the promotion calendar and price history. Success means a lower forecast error than the seasonal-naive method, especially in promotion weeks, a 4-week forecast that feeds purchase orders and smaller safety stock for the same service level.

> 💡 Think of a distribution-center planner with a whiteboard. Each week they read the new numbers, erase notes that no longer matter (forget gate), write down anything important like an upcoming promotion (input gate), and tell the buyer only the part needed for next week's order (output gate). The whiteboard is the LSTM's cell memory.

Legend in code: `# 🐍 PY:` Python lesson · `# 🧠 ML:` ML lesson

In [ ]:
# Run once (Colab already has most of these)
!pip -q install numpy pandas scikit-learn scipy xgboost torch

## CONCEPT: Step one LSTM cell through 5 weeks of demand
Compute the forget, input, candidate and output gates and the cell and hidden states by hand in numpy for five weeks of SKU demand.

| 5W + H | |
|---|---|
| **Who** | A demand planner or architect who wants to see inside an LSTM. |
| **What** | Five weeks of scaled shipments for one SKU and a single 2-unit LSTM cell with random weights. |
| **When** | Before trusting a deep-learning forecast in the replenishment process. |
| **Where** | A numpy-only script on a laptop. |
| **Why** | The gate equations are the whole idea of an LSTM; seeing their numbers week by week makes the model less of a black box. |
| **How** | Create W, U and b per gate with dict comprehensions, loop over weeks computing sigmoid and tanh gates, update c and h, then map h to an (untrained) forecast. |

**🧠 ML concepts:** Recurrent processing step by step, Gates as 0-1 switches (sigmoid), Cell state vs hidden state, Forget-gate bias trick, Output layer maps memory to a forecast  
**🐍 Python concepts:** Dict comprehensions, Defining small helper functions, Passing a function as an argument, enumerate with start=1, The @ operator for matrix and dot products, Multi-line f-strings

In [ ]:
import numpy as np

np.random.seed(42)
# 🐍 PY: weekly units shipped for one SKU, scaled to 0-1 (1.0 = 1,000 cases) - a plain list
weekly_demand = [0.42, 0.45, 0.50, 0.61, 0.58]

# 🧠 ML: an LSTM cell keeps two memories - h (short-term output) and c (long-term cell state)
hidden = 2
# 🐍 PY: np.zeros makes a vector of 0s; both memories start empty
h = np.zeros(hidden)
c = np.zeros(hidden)

# 🧠 ML: four gates, each has weights for the input x and the previous h, plus a bias
# 🐍 PY: a dict comprehension builds W (input weights), U (hidden weights) and b per gate
gates = ["forget", "input", "candidate", "output"]
W = {g: np.random.randn(hidden, 1) * 0.5 for g in gates}
U = {g: np.random.randn(hidden, hidden) * 0.5 for g in gates}
b = {g: np.zeros(hidden) for g in gates}
# 🧠 ML: forget-gate bias of +1 is a common trick - "remember by default"
b["forget"] += 1.0

# 🐍 PY: small helper functions defined with def; sigmoid squashes any number into 0..1
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def gate(name, x, h, act):
    return act(W[name] @ x + U[name] @ h + b[name])

# 🐍 PY: enumerate(..., start=1) numbers the weeks from 1 instead of 0
for week, demand in enumerate(weekly_demand, start=1):
    x = np.array([demand])
    # 🧠 ML: forget gate f - how much old memory to keep (0 = wipe, 1 = keep all)
    f = gate("forget", x, h, sigmoid)
    # 🧠 ML: input gate i - how much of the new information to write
    i = gate("input", x, h, sigmoid)
    # 🧠 ML: candidate g - the new information itself, in -1..1 via tanh
    g = gate("candidate", x, h, np.tanh)
    # 🧠 ML: output gate o - how much of the memory to reveal as this week's output
    o = gate("output", x, h, sigmoid)
    # 🧠 ML: the two key LSTM equations - update long memory, then derive the short memory
    c = f * c + i * g
    h = o * np.tanh(c)
    print(f"week {week}: demand={demand:.2f} f={np.round(f, 2)} i={np.round(i, 2)} "
          f"c={np.round(c, 3)} h={np.round(h, 3)}")

# 🧠 ML: in a real model a final linear layer turns h into next week's forecast
w_out, b_out = np.array([0.6, 0.4]), 0.55
# 🐍 PY: the @ operator between two 1-D arrays is a dot product (a single number)
forecast = (w_out @ h + b_out) * 1000
print(f"\nUntrained forecast for week 6: {forecast:.0f} cases (training would tune W, U, b)")

**Try this next:**
- Set b['forget'] to -3 and see how quickly the cell state forgets.
- Increase hidden to 4 and inspect the larger gate vectors.
- Add a promotion spike (0.95) in week 4 and watch c react.

## BUILD: Train a PyTorch LSTM on 4 years of weekly SKU demand
Train a one-layer PyTorch LSTM on sliding 12-week windows of a seasonal series and beat a naive last-week forecast on the final year.

| 5W + H | |
|---|---|
| **Who** | A supply-chain data scientist piloting deep-learning forecasts. |
| **What** | 208 weeks of synthetic shipments for one SKU with trend, yearly seasonality and noise. |
| **When** | During a pilot before scaling to many SKUs. |
| **Where** | A CPU Python environment with PyTorch; data would come from the warehouse system. |
| **Why** | To learn windowing, time-ordered splitting, scaling and the LSTM training loop on one clear series. |
| **How** | Scale with training statistics, make 12-week windows, train on the first 3 years, test on the last year, compare MAPE with last-week naive. |

**🧠 ML concepts:** Sliding-window supervised framing, Time-ordered train/test split, Scaling with training statistics only, LSTM layer and last hidden state, MAPE and a naive baseline  
**🐍 Python concepts:** Vectorized numpy maths, Function returning two tensors, Slicing with [i:i + window], Ellipsis indexing [..., None], Class with __init__ and forward, Modulo for periodic printing

In [ ]:
import numpy as np
import torch
import torch.nn as nn

torch.set_num_threads(1)
torch.manual_seed(42)
np.random.seed(42)

# 🧠 ML: 4 years of weekly demand = trend + yearly seasonality + noise (one SKU at one DC)
weeks = np.arange(208)
# 🐍 PY: numpy vector maths - every week computed at once, no loop needed
units_shipped = (800 + 0.8 * weeks + 300 * np.sin(2 * np.pi * weeks / 52)
                 + np.random.normal(0, 100, len(weeks)))
print(f"Weeks of history: {len(units_shipped)}  min={units_shipped.min():.0f}  max={units_shipped.max():.0f}")

# 🧠 ML: scale using TRAIN stats only so no information leaks from the future
split = 156
mu, sd = units_shipped[:split].mean(), units_shipped[:split].std()
scaled = (units_shipped - mu) / sd

# 🧠 ML: sliding window - use the last 12 weeks to predict the next week
WINDOW = 12
def make_windows(series, window):
    # 🐍 PY: list comprehension of slices; series[i:i+window] is 12 weeks, series[i+window] is the target
    X = [series[i:i + window] for i in range(len(series) - window)]
    y = [series[i + window] for i in range(len(series) - window)]
    # 🐍 PY: np.array(...)[..., None] adds a feature axis -> shape (samples, 12, 1)
    return torch.tensor(np.array(X)[..., None], dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

X, y = make_windows(scaled, WINDOW)
# 🧠 ML: time-ordered split - train on the past, test on the most recent year (never shuffle across time)
n_train = split - WINDOW
X_tr, y_tr, X_te, y_te = X[:n_train], y[:n_train], X[n_train:], y[n_train:]
print(f"Train windows={len(X_tr)}  Test windows={len(X_te)}")

# 🐍 PY: a subclass of nn.Module; layers are attributes, forward() defines the data flow
class DemandLSTM(nn.Module):
    def __init__(self, hidden=24):
        super().__init__()
        # 🧠 ML: batch_first=True means input shape is (batch, time steps, features)
        self.lstm = nn.LSTM(input_size=1, hidden_size=hidden, batch_first=True)
        self.out = nn.Linear(hidden, 1)

    def forward(self, x):
        seq_out, _ = self.lstm(x)
        # 🧠 ML: use only the LAST time step's hidden state as the summary of the 12 weeks
        return self.out(seq_out[:, -1, :]).squeeze(-1)

model = DemandLSTM()
opt = torch.optim.Adam(model.parameters(), lr=0.005)
loss_fn = nn.MSELoss()
for epoch in range(1, 201):
    opt.zero_grad()
    loss = loss_fn(model(X_tr), y_tr)
    loss.backward()
    opt.step()
    # 🐍 PY: the % (modulo) operator prints progress every 40 epochs
    if epoch % 40 == 0:
        print(f"epoch {epoch:3d}  train MSE={loss.item():.4f}")

model.eval()
with torch.no_grad():
    pred = model(X_te).numpy() * sd + mu
actual = y_te.numpy() * sd + mu
# 🧠 ML: naive baseline = "same as last week"; a model must beat this to be worth running
naive = X_te[:, -1, 0].numpy() * sd + mu

def mape(a, p):
    return np.mean(np.abs(a - p) / a) * 100

print(f"\nTest MAPE  LSTM={mape(actual, pred):.1f}%   naive last-week={mape(actual, naive):.1f}%")
print("\nweek  actual  lstm  naive")
# 🐍 PY: zip with range lets us print the first 8 test weeks side by side
for w, a, p, nv in zip(range(split, split + 8), actual, pred, naive):
    print(f"{w:4d} {a:7.0f} {p:5.0f} {nv:6.0f}")
print(f"\nPlanner view: average absolute error = {np.mean(np.abs(actual - pred)):.0f} cases/week")

**Try this next:**
- Change WINDOW from 12 to 52 so the model sees a full year, and compare MAPE.
- Add a seasonal-naive baseline (same week last year) and see whether the LSTM still wins.
- Double the noise level and watch both MAPEs rise.

## PRODUCTION: Multivariate 4-week forecast with promos, baseline, MAPE and safety stock
Build a multivariate, multi-step LSTM forecaster with promotions and price, tune it on a validation window, beat seasonal-naive on MAPE and translate the error into safety stock.

| 5W + H | |
|---|---|
| **Who** | The demand-planning lead, the replenishment team and the supply-chain analytics owner. |
| **What** | 260 weeks of SKU demand with holiday peaks, 15 percent promo weeks and price changes, plus the known promo calendar for the next 4 weeks. |
| **When** | Before the forecast is allowed to drive weekly purchase orders. |
| **Where** | A scheduled forecasting job; the saved model is reloaded each week for every SKU in the DC. |
| **Why** | Promotions and holidays cause the biggest stock-outs and overstock; a better forecast reduces both and lowers safety stock. |
| **How** | Five input features plus future promo flags, train/validation/test split by time, grid over hidden size and learning rate with best-epoch selection, direct 4-week output, MAPE by horizon and on promo weeks, permutation importance, safety-stock calculation and torch.save. |

**🧠 ML concepts:** Multivariate sequence inputs, Known-future covariates (planned promotions), Train/validation/test split by time, Direct multi-step forecasting, Seasonal-naive baseline and MAPE by horizon, Permutation importance and safety stock from forecast error  
**🐍 Python concepts:** Functions with docstrings and defaults, np.hstack and np.repeat to build inputs, Nested loops for a grid search, min over dict items with a key lambda, Dict comprehension mapping names to column lists, tempfile plus torch.save

In [ ]:
import os
import tempfile
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

warnings.filterwarnings("ignore")
torch.set_num_threads(1)
torch.manual_seed(42)
SEED, WINDOW, HORIZON = 42, 26, 4
FEATS = ["units", "promo", "price", "week_sin", "week_cos"]


def make_history(n_weeks=260, seed=SEED):
    """5 years of weekly demand for one SKU at one DC, with promotions and price changes."""
    rng = np.random.default_rng(seed)
    w = np.arange(n_weeks)
    # 🐍 PY: (array < 0.15).astype(int) turns True/False into 1/0 promo flags
    promo = (rng.random(n_weeks) < 0.15).astype(int)
    # 🧠 ML: promos cut price by ~2 dollars, so price and promo are correlated inputs
    price = np.round(12 - 2.0 * promo + rng.normal(0, 0.3, n_weeks), 2)
    season = 250 * np.sin(2 * np.pi * w / 52) + 120 * (w % 52 >= 47)    # holiday peak
    # 🧠 ML: promo lifts demand ~35% and price elasticity lowers it - the LSTM must learn both
    units = (900 + 0.8 * w + season) * (1 + 0.35 * promo) * (price / 12) ** -1.2
    units = units + rng.normal(0, 70, n_weeks)
    df = pd.DataFrame({"week": w, "units": units.round(), "promo": promo, "price": price})
    # 🧠 ML: sine/cosine of week-of-year lets the model know where it is in the seasonal cycle
    df["week_sin"] = np.sin(2 * np.pi * w / 52)
    df["week_cos"] = np.cos(2 * np.pi * w / 52)
    return df


def windows(arr, future_promo, start, end):
    """X = last WINDOW weeks of all features + KNOWN future promo flags; y = next HORIZON weeks."""
    X, y = [], []
    for t in range(max(start, WINDOW), end - HORIZON + 1):
        # 🐍 PY: slice [t - WINDOW:t] grabs the 26 rows before week t, all feature columns
        past = arr[t - WINDOW:t]
        # 🧠 ML: planned promos for the next 4 weeks are known in advance - feed them in
        fut = np.repeat(future_promo[t:t + HORIZON][None, :], WINDOW, axis=0)
        X.append(np.hstack([past, fut]))
        # 🧠 ML: target = the next 4 weeks of scaled units (column 0)
        y.append(arr[t:t + HORIZON, 0])
    return torch.tensor(np.array(X), dtype=torch.float32), torch.tensor(np.array(y), dtype=torch.float32)


class Forecaster(nn.Module):
    def __init__(self, n_in, hidden=32, layers=1, dropout=0.0):
        super().__init__()
        self.lstm = nn.LSTM(n_in, hidden, num_layers=layers, batch_first=True)
        self.drop = nn.Dropout(dropout)
        # 🧠 ML: direct multi-step - one head outputs all 4 future weeks at once (no error feedback loop)
        self.head = nn.Linear(hidden, HORIZON)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(self.drop(out[:, -1]))


def fit(Xtr, ytr, Xva, yva, hidden, lr, epochs=150):
    # 🧠 ML: reseeding makes every grid candidate start from comparable random weights
    torch.manual_seed(SEED)
    m = Forecaster(Xtr.shape[2], hidden, dropout=0.1)
    opt = torch.optim.Adam(m.parameters(), lr=lr)
    best, state = 1e9, None
    for _ in range(epochs):
        m.train()
        opt.zero_grad()
        # 🧠 ML: full-batch training is fine here - only ~150 windows
        nn.functional.mse_loss(m(Xtr), ytr).backward()
        opt.step()
        m.eval()
        with torch.no_grad():
            v = nn.functional.mse_loss(m(Xva), yva).item()
        # 🧠 ML: keep weights from the best validation epoch (early stopping)
        if v < best:
            best, state = v, {k: t.clone() for k, t in m.state_dict().items()}
    m.load_state_dict(state)
    return m, best


# 🧠 ML: MAPE = average absolute % miss; planners understand it, but it explodes near zero demand
def mape(a, p):
    return float(np.mean(np.abs(a - p) / np.abs(a)) * 100)


df = make_history()
n = len(df)
# 🧠 ML: time-ordered split: train weeks 0-181 | validate 182-207 | test last 52 weeks (never shuffle)
tr_end, va_end = n - 78, n - 52
# 🧠 ML: scale using training weeks only to avoid leaking future level into the model
mu, sd = df[FEATS][:tr_end].mean(), df[FEATS][:tr_end].std()
arr = ((df[FEATS] - mu) / sd).values
Xtr, ytr = windows(arr, df.promo.values, 0, tr_end)
Xva, yva = windows(arr, df.promo.values, tr_end, va_end)
Xte, yte = windows(arr, df.promo.values, va_end, n)
print(f"windows train/val/test = {len(Xtr)}/{len(Xva)}/{len(Xte)}  inputs per step = {Xtr.shape[2]}")

# 🧠 ML: small grid over hidden size and learning rate, chosen on the validation window
results = {}
for hidden in [16, 32]:
    for lr in [0.01, 0.003]:
        model, v = fit(Xtr, ytr, Xva, yva, hidden, lr)
        results[(hidden, lr)] = (v, model)
        print(f"  hidden={hidden:2d} lr={lr:<6} val MSE={v:.3f}")
# 🐍 PY: min with key=lambda picks the dict item whose validation MSE (kv[1][0]) is smallest
(best_h, best_lr), (_, model) = min(results.items(), key=lambda kv: kv[1][0])
print(f"Chosen hidden={best_h} lr={best_lr}")

model.eval()
with torch.no_grad():
    # 🧠 ML: undo scaling so errors are in real cases, not z-scores
    pred = model(Xte).numpy() * sd["units"] + mu["units"]
actual = yte.numpy() * sd["units"] + mu["units"]
# 🧠 ML: seasonal-naive baseline = same week last year (hard to beat for seasonal SKUs)
starts = np.arange(max(va_end, WINDOW), n - HORIZON + 1)
# 🐍 PY: list comprehension slices the same 4 weeks from 52 weeks earlier for each forecast start
snaive = np.array([df.units.values[s - 52:s - 52 + HORIZON] for s in starts])

print("\nMAPE by horizon   LSTM   seasonal-naive")
for h in range(HORIZON):
    print(f"  week +{h + 1}        {mape(actual[:, h], pred[:, h]):5.1f}%   {mape(actual[:, h], snaive[:, h]):5.1f}%")
print(f"  overall        {mape(actual, pred):5.1f}%   {mape(actual, snaive):5.1f}%")

# 🧠 ML: promo weeks are where seasonal-naive fails - check accuracy split by promo flag
promo_mask = np.array([df.promo.values[s] for s in starts]) == 1
print(f"Week+1 MAPE on promo weeks: LSTM {mape(actual[promo_mask, 0], pred[promo_mask, 0]):.1f}%"
      f" vs naive {mape(actual[promo_mask, 0], snaive[promo_mask, 0]):.1f}%")

# 🧠 ML: permutation importance on test - shuffle one input feature, see how much MAPE worsens
base = mape(actual, pred)
rng = np.random.default_rng(SEED)
print("\nFeature importance (MAPE increase when shuffled):")
# 🐍 PY: a dict maps a readable name to the list of input columns that belong to it
groups = {name: [j] for j, name in enumerate(FEATS)}
groups["future_promo"] = list(range(len(FEATS), Xte.shape[2]))
for name, cols in groups.items():
    Xp = Xte.clone()
    Xp[:, :, cols] = Xp[rng.permutation(len(Xp))][:, :, cols]
    with torch.no_grad():
        p2 = model(Xp).numpy() * sd["units"] + mu["units"]
    print(f"  {name:12s} {mape(actual, p2) - base:+5.1f} pts")

# 🧠 ML: safety stock = z * sigma of forecast error * sqrt(lead time); z=1.65 -> 95% service level
lead_weeks = 2
for label, p in [("LSTM", pred), ("seasonal-naive", snaive)]:
    # 🧠 ML: sigma of week+1 forecast error drives how much buffer stock is needed
    sigma = np.std(actual[:, 0] - p[:, 0])
    ss = 1.65 * sigma * np.sqrt(lead_weeks)
    print(f"Safety stock ({label:14s}): sigma={sigma:5.0f}  -> {ss:5.0f} units at 95% service")

print("\nNext 4-week forecast:", [int(x) for x in pred[-1]], " actual:", [int(x) for x in actual[-1]])
# Governance: forecasts drive purchase orders; log model version, monitor MAPE drift weekly,
# and let planners override with a reason code (S&OP process).
print("Governance: version the model, monitor weekly MAPE drift, log planner overrides in S&OP.")
# 🧠 ML: save scaling stats with the weights so next week's data is scaled identically
path = os.path.join(tempfile.mkdtemp(), "sku_lstm.pt")
torch.save({"state": model.state_dict(), "mu": mu.to_dict(), "sd": sd.to_dict()}, path)
print("Saved ->", os.path.basename(path))

**Try this next:**
- Remove the future promo flags from windows (skip np.hstack) and measure how much promo-week MAPE gets worse.
- Change lead_weeks to 4 and see the safety-stock impact.
- Add a 2-layer LSTM (layers=2) to the grid and check whether validation improves.